# Gomi — Full Training Pipeline

**Step 1:** Fine-tune DistilBERT on combined OpenReview dataset (2k human + 5k auto-labeled)  
**Step 2:** Train Logistic Regression risk fusion model on DeepJIT + ApacheJIT  
**Step 3:** Push both models to Hugging Face Hub

> ⚡ Make sure **Runtime → Change runtime type → T4 GPU** is selected before running.

---
**Hugging Face repos:**
- Dataset: `GitRatBCSAD/gomi-datasets`
- Sentiment model: `GitRatBCSAD/gomi-sentiment`
- Risk model: `GitRatBCSAD/gomi-risk`

In [ ]:
# ── 0. Check GPU ──────────────────────────────────────────────────────────────
import torch
import torchvision.io
class _VideoReaderStub:
    pass

torchvision.io.VideoReader = _VideoReaderStub

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')


In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q transformers==4.52.4 datasets scikit-learn huggingface_hub joblib shap lizard

In [ ]:
# ── 2. Authenticate with Hugging Face ─────────────────────────────────────────
# Paste your HF token (Settings → Access Tokens → New token → Write)
from google.colab import userdata
from huggingface_hub import login
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print('Logged in to Hugging Face.')

In [ ]:
# ── 3. Config ─────────────────────────────────────────────────────────────────
HF_DATASET_REPO   = 'GitRatBCSAD/gomi-datasets'
HF_SENTIMENT_REPO = 'GitRatBCSAD/gomi-sentiment'
HF_RISK_REPO      = 'GitRatBCSAD/gomi-risk'

BASE_MODEL     = 'distilbert-base-uncased'
LABELS         = ['frustration', 'caution', 'neutral', 'satisfaction']
LABEL2ID       = {l: i for i, l in enumerate(LABELS)}
ID2LABEL       = {i: l for i, l in enumerate(LABELS)}
VALID_EMOTIONS = set(LABELS)
RISK_LABELS    = {'frustration', 'caution'}

# Training hyperparameters
MAX_LENGTH     = 128
BATCH_SIZE     = 32    # good for T4 with DistilBERT
NUM_EPOCHS     = 5
LEARNING_RATE  = 2e-5
WEIGHT_DECAY   = 0.01
TEST_SIZE      = 0.15
RANDOM_SEED    = 42
MIN_CONFIDENCE = 0.90  # filter threshold for 5k auto-labeled rows

import os
os.makedirs('models/distilbert_sentiment', exist_ok=True)
os.makedirs('models/risk', exist_ok=True)
print('Config done.')

---
## Step 1 — Fine-tune DistilBERT
Training on **2k human-labeled + 5k auto-labeled** commits (~7k total).

In [ ]:
# ── 4. Load datasets from HuggingFace ─────────────────────────────────────────
import csv
from collections import Counter
from huggingface_hub import hf_hub_download

def load_labeled_csv(filename, min_confidence=None):
    path = hf_hub_download(HF_DATASET_REPO, f'openreview/{filename}', repo_type='dataset', token=HF_TOKEN)
    messages, label_ids = [], []
    skipped = low_conf = 0
    with open(path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            msg     = row.get('message', '').strip()
            emotion = row.get('reconciled_emotion', '').strip().lower()
            if not msg or emotion not in VALID_EMOTIONS:
                skipped += 1; continue
            if min_confidence is not None and 'confidence' in row:
                try:
                    if float(row['confidence']) < min_confidence:
                        low_conf += 1; continue
                except (ValueError, TypeError):
                    pass
            messages.append(msg)
            label_ids.append(LABEL2ID[emotion])
    note = f', {low_conf} below conf {min_confidence}' if low_conf else ''
    print(f'  {filename}: {len(messages)} rows ({skipped} skipped{note})')
    dist = Counter(LABELS[i] for i in label_ids)
    for lbl, cnt in sorted(dist.items()): print(f'    {lbl:<14} {cnt:>4}  ({100*cnt/len(label_ids):.1f}%)')
    return messages, label_ids

print('[1/4] Loading datasets...')
print('  [2k human-labeled]')
msgs_2k, ids_2k = load_labeled_csv('openreview_labeled_2k.csv')

print('  [1.6k SentiCR human-labeled]')
msgs_scr, ids_scr = load_labeled_csv('senticr_labeled.csv')

# 
messages  = msgs_2k + msgs_scr
label_ids = ids_2k + ids_scr
print(f"
print(f"\n  Combined: {len(messages)} samples ({len(msgs_2k)} human + {len(msgs_scr)} SentiCR)")

In [ ]:
# ── 5. Tokenize & split ────────────────────────────────────────────────────────
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset

train_msgs, val_msgs, train_labels, val_labels = train_test_split(
    messages, label_ids, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=label_ids
)
print(f'[2/4] Split: {len(train_msgs)} train | {len(val_msgs)} val')

print(f'      Loading tokenizer ({BASE_MODEL})...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

train_ds = Dataset.from_dict({'text': train_msgs, 'label': train_labels}).map(tokenize, batched=True).remove_columns(['text'])
val_ds   = Dataset.from_dict({'text': val_msgs,   'label': val_labels  }).map(tokenize, batched=True).remove_columns(['text'])
train_ds.set_format('torch')
val_ds.set_format('torch')
collator = DataCollatorWithPadding(tokenizer=tokenizer)
print('      Tokenization done.')

In [ ]:
# ── 6. Fine-tune ───────────────────────────────────────────────────────────────
from sklearn.metrics import classification_report

print(f'[3/4] Loading {BASE_MODEL} and attaching classification head...')
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds  = np.argmax(logits, axis=-1)
    report = classification_report(labels, preds, target_names=LABELS, output_dict=True, zero_division=0)
    return {'accuracy': report['accuracy'], 'f1_macro': report['macro avg']['f1-score'],
            'precision': report['macro avg']['precision'], 'recall': report['macro avg']['recall']}

args = TrainingArguments(
    output_dir='models/distilbert_sentiment/checkpoints',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=20,
    report_to='none',
    seed=RANDOM_SEED,
    fp16=torch.cuda.is_available(),   # use mixed precision on GPU
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics,
)

print(f'[4/4] Fine-tuning for {NUM_EPOCHS} epochs on {len(train_msgs)} samples...')
trainer.train()

In [ ]:
# ── 7. Evaluate & save locally ────────────────────────────────────────────────
print('
Final evaluation on validation set:')
preds_out = trainer.predict(val_ds)
preds     = np.argmax(preds_out.predictions, axis=-1)
print(classification_report(val_labels, preds, target_names=LABELS, zero_division=0))

SENTIMENT_MODEL_DIR = 'models/distilbert_sentiment'
model.save_pretrained(SENTIMENT_MODEL_DIR)
tokenizer.save_pretrained(SENTIMENT_MODEL_DIR)
print(f'
Model saved to: {SENTIMENT_MODEL_DIR}')

In [ ]:
# ── 8. Push DistilBERT to HuggingFace ─────────────────────────────────────────
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id=HF_SENTIMENT_REPO, repo_type='model', exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    repo_id=HF_SENTIMENT_REPO, repo_type='model',
    folder_path=SENTIMENT_MODEL_DIR, path_in_repo='.', token=HF_TOKEN,
)
print(f'Uploaded to: https://huggingface.co/{HF_SENTIMENT_REPO}')

---
## Step 2 — Train Logistic Regression Risk Model
Uses the fine-tuned DistilBERT above to generate sentiment features for DeepJIT commits.

In [ ]:
# ── 9. Load fine-tuned sentiment classifier ───────────────────────────────────
import re
from transformers import pipeline as hf_pipeline

CONVENTIONAL_COMMIT_RE  = r'^[a-z]+(\([^)]+\))?!?:\s*'
LOW_INFO_TOKEN_THRESHOLD = 5

def strip_prefix(msg):
    return re.sub(CONVENTIONAL_COMMIT_RE, '', msg or '', flags=re.IGNORECASE).strip()

def is_low_info(msg):
    return len(strip_prefix(msg).split()) < LOW_INFO_TOKEN_THRESHOLD

device = 0 if torch.cuda.is_available() else -1
sentiment_clf = hf_pipeline(
    'text-classification',
    model=SENTIMENT_MODEL_DIR,
    tokenizer=SENTIMENT_MODEL_DIR,
    top_k=None, truncation=True, max_length=128,
    device=device,
)
print(f'Sentiment classifier loaded (device={"GPU" if device==0 else "CPU"}).')

def classify_batch(messages):
    """Batch inference — returns list of labels."""
    cleaned = [strip_prefix(m)[:512] or 'empty' for m in messages]
    results = sentiment_clf(cleaned, batch_size=64)
    labels  = []
    for r in results:
        best  = max(r, key=lambda x: x['score'])
        label = best['label'].lower().replace('label_', '')
        if label not in VALID_EMOTIONS:
            label = next((k for k in VALID_EMOTIONS if k in label), 'neutral')
        labels.append(label)
    return labels

In [ ]:
# ── 10. Load DeepJIT (All 6 Projects) ─────────────────────────────────────────
import pickle
import os

def percentile_rank(value, all_values):
    if not all_values or len(all_values) == 1: return 0.0
    return round(sum(1 for x in all_values if x <= value) / len(all_values), 4)

DEEPJIT_PKLS = [
    "qt_test_raw.pkl",
    "openstack_test_raw.pkl",
    "go_test_raw.pkl",
    "jdt_test_raw.pkl",
    "gerrit_test_raw.pkl",
    "platform_test_raw.pkl",
]

deepjit_records = []

for pkl_file in DEEPJIT_PKLS:
    feat_file = pkl_file.replace("_test_raw.pkl", "_k_feature.csv")
    proj = pkl_file.split("_")[0]
    print(f"
Loading DeepJIT ({proj})...")

    try:
        pkl_path = hf_hub_download(HF_DATASET_REPO, f"jit/{pkl_file}", repo_type="dataset", token=HF_TOKEN)
        feat_path = hf_hub_download(HF_DATASET_REPO, f"jit/{feat_file}", repo_type="dataset", token=HF_TOKEN)
    except Exception as e:
        print(f"  [skip] {proj} missing on HF: {e}")
        continue

    with open(pkl_path, "rb") as f:
        raw = pickle.load(f)
    hashes, labels_jit, messages_jit = raw[0], raw[1], raw[2]

    ent_by_hash = {}
    with open(feat_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            try: ent_by_hash[row["_id"]] = float(row["entrophy"])
            except (ValueError, KeyError): pass

    all_ent = list(ent_by_hash.values())
    print(f"  {len(labels_jit)} commits, entropy available for {len(all_ent)}")

    # Batch DistilBERT inference
    low_info_flags = [is_low_info(m) for m in messages_jit]
    non_low_msgs   = [m for m, li in zip(messages_jit, low_info_flags) if not li]
    non_low_labels = classify_batch(non_low_msgs)

    # Rebuild full label list
    jit_sentiment_labels = []
    nli = 0
    for li in low_info_flags:
        if li: jit_sentiment_labels.append("neutral")
        else:  jit_sentiment_labels.append(non_low_labels[nli]); nli += 1

    for h, msg, label, sent_label, li in zip(hashes, messages_jit, labels_jit, jit_sentiment_labels, low_info_flags):
        sent_score = 1.0 if sent_label in RISK_LABELS else 0.0
        comp_score = percentile_rank(ent_by_hash.get(h, 0.0), all_ent) if h in ent_by_hash else 0.5
        deepjit_records.append({
            'sentiment_score': sent_score,
            'complexity_score': comp_score,
            'low_info_ratio': 1.0 if li else 0.0,
            'buggy': int(label),
        })

print(f"
Total DeepJIT cross-project commits loaded: {len(deepjit_records)}")


In [ ]:
# ── 11. Load ApacheJIT (test_small — has features, no commit messages) ────────
# apachejit_test_small.csv has complexity features but no 'message' column.
# We use it as validation-only (ground truth for Precision/Recall/F1).
# For training we rely on Qt DeepJIT which has commit messages for DistilBERT.

print('Loading ApacheJIT (validation split)...')
apache_path = hf_hub_download(HF_DATASET_REPO, 'jit/apachejit_test_small.csv', repo_type='dataset', token=HF_TOKEN)

apache_val_records = []
with open(apache_path, newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        try:
            buggy = 1 if str(row.get('buggy', 'False')).lower() in ('true', '1') else 0
            ent   = float(row.get('ent', 0.5))
            apache_val_records.append({'ent': ent, 'buggy': buggy})
        except (ValueError, KeyError):
            continue

# Percentile-rank entropy within ApacheJIT for normalization
all_apache_ent = [r['ent'] for r in apache_val_records]
apache_val_feat = []
for r in apache_val_records:
    apache_val_feat.append({
        'sentiment_score': 0.5,           # no messages → neutral proxy
        'complexity_score': percentile_rank(r['ent'], all_apache_ent),
        'low_info_ratio': 0.0,
        'buggy': r['buggy'],
    })
print(f'ApacheJIT validation: {len(apache_val_feat)} commits')
buggy_n = sum(r["buggy"] for r in apache_val_feat)
print(f'  {buggy_n} buggy ({100*buggy_n/len(apache_val_feat):.1f}%), {len(apache_val_feat)-buggy_n} clean')

In [ ]:
# ── 12. Train Logistic Regression ────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report as clf_report

all_train = deepjit_records
X_train = np.array([[r['sentiment_score'], r['complexity_score'], r['low_info_ratio']] for r in all_train])
y_train = np.array([r['buggy'] for r in all_train])

risk_model = LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000)
risk_model.fit(X_train, y_train)

buggy_n = int(y_train.sum())
print(f'Training set: {len(y_train)} commits ({buggy_n} buggy, {len(y_train)-buggy_n} clean)')
print(f'LR coef → sentiment: {risk_model.coef_[0][0]:.4f}  '
      f'complexity: {risk_model.coef_[0][1]:.4f}  '
      f'low_info: {risk_model.coef_[0][2]:.4f}')
print(f'Intercept: {risk_model.intercept_[0]:.4f}')

# Validate on ApacheJIT
X_val = np.array([[r['sentiment_score'], r['complexity_score'], r['low_info_ratio']] for r in apache_val_feat])
y_val = np.array([r['buggy'] for r in apache_val_feat])
preds = risk_model.predict(X_val)
print('
Validation (ApacheJIT held-out):')
print(clf_report(y_val, preds, target_names=['clean', 'buggy'], zero_division=0))

In [ ]:
# ── 13. Save risk model + SHAP background ────────────────────────────────────
import joblib

RISK_MODEL_PATH = 'models/risk/risk_model.joblib'
SHAP_BG_PATH    = 'models/risk/risk_model_shap_background.npy'

joblib.dump(risk_model, RISK_MODEL_PATH)
np.save(SHAP_BG_PATH, X_train)
print(f'Saved: {RISK_MODEL_PATH}')
print(f'Saved: {SHAP_BG_PATH}  (shape: {X_train.shape})')

In [ ]:
# ── 14. Push risk model to HuggingFace ───────────────────────────────────────
api.create_repo(repo_id=HF_RISK_REPO, repo_type='model', exist_ok=True, token=HF_TOKEN)
api.upload_file(path_or_fileobj=RISK_MODEL_PATH,
                path_in_repo='risk_model.joblib',
                repo_id=HF_RISK_REPO, repo_type='model', token=HF_TOKEN)
api.upload_file(path_or_fileobj=SHAP_BG_PATH,
                path_in_repo='risk_model_shap_background.npy',
                repo_id=HF_RISK_REPO, repo_type='model', token=HF_TOKEN)
print(f'Uploaded to: https://huggingface.co/{HF_RISK_REPO}')

In [ ]:
# ── 15. Download trained models back locally (optional) ───────────────────────
# Run this if you want to download the trained artifacts to your local machine.
# Otherwise, gomi.py will pull them from HuggingFace automatically at runtime
# via GOMI_SENTIMENT_MODEL_REPO and GOMI_RISK_MODEL_REPO in your .env.

from google.colab import files
import shutil

# Zip the sentiment model
shutil.make_archive('distilbert_sentiment', 'zip', 'models/distilbert_sentiment')
files.download('distilbert_sentiment.zip')

# Download risk model files individually
files.download(RISK_MODEL_PATH)
files.download(SHAP_BG_PATH)
print('Downloads started.')

---
## Done!

Both models are now on HuggingFace:
- **DistilBERT sentiment:** `GitRatBCSAD/gomi-sentiment`
- **Risk model (LR + SHAP):** `GitRatBCSAD/gomi-risk`

Your `.env` already points to these repos, so `gomi.py` will pull them at runtime automatically.

```
GOMI_SENTIMENT_MODEL_REPO=GitRatBCSAD/gomi-sentiment
GOMI_RISK_MODEL_REPO=GitRatBCSAD/gomi-risk
```

To run Gomi locally after training:
```bash
python gomi.py /path/to/your/repo
```